# Setup
Revisar `README.md` para mayor información sobre requerimientos de reproducibilidad del presente notebook.

Creamos el entorno.

In [1]:
import os, shutil, subprocess
from pathlib import Path

ROOT = Path.cwd()
VENV = ROOT / ".venv"
KERNEL_NAME = "recsys-obp"
KERNEL_DISPLAY = "Python 3.10 (recsys-obp)"
venv_python = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(cmd):
    cmd = [str(c) for c in cmd]
    print(">", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f"Falló: {' '.join(cmd)}")


def find_python310():
    candidates = []
    if os.environ.get("PYTHON310"):
        candidates.append([os.environ["PYTHON310"]])
    if shutil.which("py"):
        candidates.append(["py", "-3.10"])
    for name in ("python3.10", "python3.10.exe"):
        if shutil.which(name):
            candidates.append([shutil.which(name)])
    for base in (os.environ.get("LOCALAPPDATA", ""), os.environ.get("ProgramFiles", "")):
        exe = Path(base) / "Programs" / "Python" / "Python310" / "python.exe"
        for path in (exe, Path(base) / "Python310" / "python.exe"):
            if path.exists():
                candidates.append([str(path)])
    for cmd in candidates:
        try:
            out = subprocess.run(cmd + ["-c", "import sys; print(sys.version_info[:2])"],
                                 capture_output=True, text=True)
            if out.returncode == 0 and out.stdout.strip() == "(3, 10)":
                return cmd
        except OSError:
            pass
    raise RuntimeError(
        "No se encontró Python 3.10. Instálalo desde https://www.python.org/downloads/ "
        "(versión 3.10.x) o define la variable de entorno PYTHON310 con la ruta a su python.exe."
    )


if not venv_python.exists():
    run(find_python310() + ["-m", "venv", VENV])
run([venv_python, "-m", "pip", "install", "--upgrade", "pip"])
run([venv_python, "-m", "pip", "install", "-r", ROOT / "requirements.txt"])
run([venv_python, "-m", "ipykernel", "install", "--user",
     "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY])
print(f"\nListo. Cambia el kernel a '{KERNEL_DISPLAY}' y reinícialo.")

> /Users/josita/Documents/uni/2026-2/RecSys/OPE-Proyecto-recsys-1/.venv/bin/python -m pip install --upgrade pip
> /Users/josita/Documents/uni/2026-2/RecSys/OPE-Proyecto-recsys-1/.venv/bin/python -m pip install -r /Users/josita/Documents/uni/2026-2/RecSys/OPE-Proyecto-recsys-1/requirements.txt
> /Users/josita/Documents/uni/2026-2/RecSys/OPE-Proyecto-recsys-1/.venv/bin/python -m ipykernel install --user --name recsys-obp --display-name Python 3.10 (recsys-obp)

Listo. Cambia el kernel a 'Python 3.10 (recsys-obp)' y reinícialo.


Verificamos el entorno creado e importamos las librerías necesarias:

In [1]:
import sys
assert sys.version_info[:2] == (3, 10), (
    f"Kernel equivocado (Python {sys.version.split()[0]}). "
    "Cambia al kernel 'Python 3.10 (recsys-obp)'."
)

import obp, numpy, pandas, sklearn, scipy, matplotlib, torch, pyarrow
for mod in (obp, numpy, pandas, sklearn, scipy, matplotlib, torch, pyarrow):
    print(f"{mod.__name__:<12} {mod.__version__}")

c:\Users\ddono\Desktop\2026-2\RecSys\Proyecto\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


obp          0.5.7
numpy        1.23.5
pandas       1.5.2
sklearn      1.1.3
scipy        1.9.3
matplotlib   3.8.4
torch        1.12.0+cpu
pyarrow      25.0.1


Verificamos la inclusión de datasets en las rutas respectivas

In [2]:
from pathlib import Path

DATA = Path("Contenido/Datasets")
OBD_PATH = DATA / "open_bandit_dataset" / "open_bandit_dataset"
KUAI_PATH = DATA / "KuaiRand-Pure" / "data"

for p in (OBD_PATH / "random" / "men" / "men.csv",
          OBD_PATH / "bts" / "men" / "men.csv",
          KUAI_PATH / "log_random_4_22_to_5_08_pure.csv"):
    print("OK   " if p.exists() else "Falta incluir datasets o modificar la ruta acorde a README.md", p)

OK    Contenido\Datasets\open_bandit_dataset\open_bandit_dataset\random\men\men.csv
OK    Contenido\Datasets\open_bandit_dataset\open_bandit_dataset\bts\men\men.csv
OK    Contenido\Datasets\KuaiRand-Pure\data\log_random_4_22_to_5_08_pure.csv


Verificamos el acceso y carga de datos asociados a los datasets cargados

In [3]:
from obp.dataset import OpenBanditDataset

ds = OpenBanditDataset(behavior_policy="random", campaign="men", data_path=OBD_PATH)
fb = ds.obtain_batch_bandit_feedback()
print(f"{fb['n_rounds']:,} filas | {fb['n_actions']} acciones | {ds.len_list} posiciones | CTR = {fb['reward'].mean():.4f}")

452,949 filas | 34 acciones | 3 posiciones | CTR = 0.0051
